In [5]:
import math
import time

import mujoco
import mujoco.viewer
import numpy as np


XML_PATH = "../assets/test_scene.xml"
BODY_NAME = "tracked_object"

sample_requested = False
sample_center = np.array([0.0, 0.0, 0.035], dtype=float)

STEP_XY = 0.05
STEP_Z = 0.02


# -------------------------
# Keyboard
# -------------------------
KEY_RIGHT = 262
KEY_LEFT = 263
KEY_DOWN = 264
KEY_UP = 265


def key_callback(keycode):
    global sample_requested, sample_center

    if keycode == ord(" "):
        sample_requested = True

    elif keycode == KEY_UP:
        sample_center[1] += STEP_XY
    elif keycode == KEY_DOWN:
        sample_center[1] -= STEP_XY
    elif keycode == KEY_LEFT:
        sample_center[0] -= STEP_XY
    elif keycode == KEY_RIGHT:
        sample_center[0] += STEP_XY

    elif keycode == ord("Z"):
        sample_center[2] += STEP_Z
    elif keycode == ord("X"):
        sample_center[2] = max(0.0, sample_center[2] - STEP_Z)

    elif keycode == ord("R"):
        sample_center[:] = np.array([0.0, 0.0, 0.035])

    print(f"sample_center = {sample_center}")


# -------------------------
# MuJoCo helpers
# -------------------------

def get_body_qpos_addr(model, body_name):
    body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body_name)
    joint_id = model.body_jntadr[body_id]
    return model.jnt_qposadr[joint_id]


def set_pose(data, qpos_adr, pos, quat):
    data.qpos[qpos_adr:qpos_adr + 3] = pos
    data.qpos[qpos_adr + 3:qpos_adr + 7] = quat


def yaw_to_quat(yaw):
    return np.array([
        math.cos(yaw / 2),
        0.0,
        0.0,
        math.sin(yaw / 2),
    ])


def update_sample_center_site(model, data, site_id):
    model.site_pos[site_id] = sample_center
    mujoco.mj_forward(model, data)


# -------------------------
# Sampling
# -------------------------

def sample_candidates(center, num=30, radius_xy=0.10):
    samples = []

    for _ in range(num):
        pos = np.array([
            center[0] + np.random.uniform(-radius_xy, radius_xy),
            center[1] + np.random.uniform(-radius_xy, radius_xy),
            center[2],
        ])

        yaw = np.random.uniform(-math.pi, math.pi)
        quat = yaw_to_quat(yaw)

        samples.append((pos, quat))

    return samples


# -------------------------
# Validation
# -------------------------

def validate_candidate(model, data, qpos_adr, pos, quat):
    original_qpos = data.qpos.copy()
    original_qvel = data.qvel.copy()

    set_pose(data, qpos_adr, pos, quat)
    data.qvel[:] = 0.0
    mujoco.mj_forward(model, data)

    for _ in range(100):
        mujoco.mj_step(model, data)

    final_pos = data.qpos[qpos_adr:qpos_adr + 3].copy()

    valid = True

    # Fell below table / unstable
    if final_pos[2] < 0.01:
        valid = False

    # Drifted too far from requested pose
    if np.linalg.norm(final_pos[:2] - pos[:2]) > 0.05:
        valid = False

    data.qpos[:] = original_qpos
    data.qvel[:] = original_qvel
    mujoco.mj_forward(model, data)

    return valid


# -------------------------
# Main
# -------------------------

def main():
    global sample_requested

    model = mujoco.MjModel.from_xml_path(XML_PATH)
    data = mujoco.MjData(model)

    qpos_adr = get_body_qpos_addr(model, BODY_NAME)
    site_id = mujoco.mj_name2id(
        model,
        mujoco.mjtObj.mjOBJ_SITE,
        "sample_center",
    )

    print("Controls:")
    print("  ↑ ↓ ← → : move sample center in XY")
    print("  A / S   : move sample center up/down")
    print("  R       : reset sample center")
    print("  SPACE   : sample + validate poses")
    with mujoco.viewer.launch_passive(
        model,
        data,
        key_callback=key_callback,
    ) as viewer:

        while viewer.is_running():
            update_sample_center_site(model, data, site_id)

            if sample_requested:
                sample_requested = False

                candidates = sample_candidates(sample_center, num=30)

                valid = []
                for pos, quat in candidates:
                    if validate_candidate(model, data, qpos_adr, pos, quat):
                        valid.append((pos, quat))

                print(f"Valid: {len(valid)} / {len(candidates)}")

                if valid:
                    best_pos, best_quat = valid[0]
                    set_pose(data, qpos_adr, best_pos, best_quat)
                    data.qvel[:] = 0.0
                    mujoco.mj_forward(model, data)

            mujoco.mj_step(model, data)
            viewer.sync()
            time.sleep(0.01)


if __name__ == "__main__":
    main()

Controls:
  ↑ ↓ ← → : move sample center in XY
  A / S   : move sample center up/down
  R       : reset sample center
  SPACE   : sample + validate poses
sample_center = [0.    0.05  0.035]
sample_center = [0.    0.1   0.035]
sample_center = [0.    0.05  0.035]
sample_center = [0.    0.    0.035]
sample_center = [-0.05   0.     0.035]
sample_center = [-0.1    0.     0.035]
sample_center = [-0.05   0.     0.035]
sample_center = [0.    0.    0.035]
sample_center = [0.    0.    0.055]
sample_center = [0.    0.    0.075]
sample_center = [0.    0.    0.055]
sample_center = [0.    0.    0.035]
sample_center = [0.    0.    0.015]
sample_center = [0. 0. 0.]
sample_center = [0.   0.   0.02]
sample_center = [0.   0.   0.04]
sample_center = [0.   0.   0.06]
sample_center = [0.   0.   0.08]
sample_center = [0.  0.  0.1]
sample_center = [0.   0.   0.08]
sample_center = [0.   0.   0.06]
sample_center = [0.   0.   0.04]
sample_center = [0.   0.   0.02]
sample_center = [0.   0.05 0.02]
sample_center =